# 01 – Data collection

Scrapes CS2 item listings from pricempire.com, the item pages, and the price history of each item from the site's internal chart API.

With `test = True` the notebook only processes a small sample (four listing pages of knives and ten item pages), so it runs in a few minutes. The full run (`test = False`) takes about seven hours, because requests are deliberately spaced out.

Output: `final.csv` (or `test_final.csv` in test mode), the input of `02_analysis.ipynb`.

In [ ]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
import math
import time
import os
import gc
import ast
import json
import pandas as pd
import duckdb
from tqdm import tqdm

test = True #Set to True to run the whole pipeline on a small sample
prefix = "test_" if test else "" #Output files of a test run get a "test_" prefix

## 1. Listing pages

In [ ]:
def get_skin_html_metadata(): #Scrape the listing pages of each item category

    htmls = [] #Output list: [item category, page html]
    base = [ #Listing URL of each item category, split around the page number
        ["knife",["https://pricempire.com/cs2-skin-search?page=",str(1),"&limit=96&type=skin&category=knives&rarity=covert&grouped=false"]],
        ["gun",["https://pricempire.com/cs2-skin-search?page=",str(1),"&limit=96&type=skin&category=smgs,rifles,pistols,heavy&rarity=covert&grouped=false"]],
        ["case",["https://pricempire.com/cs2-skin-search?page=",str(1),"&limit=96&type=container&grouped=false"]],
        ["key",["https://pricempire.com/cs2-skin-search?page=",str(1),"&limit=96&type=key&grouped=false"]]
    ]

    if (test): #In test mode only scrape knives
        base = [base[0]]

    driver = webdriver.Chrome() #Start a Chrome webdriver (the pages are rendered with JavaScript)

    for good in base: #Iterate through the item categories
        driver.get(good[1][0]+good[1][1]+good[1][2]) #Open the first listing page
        pages = range(1,1+max([int(x.text) for x in driver.find_elements(By.CSS_SELECTOR,".px-3.transition-colors")])) #Read the number of pages from the pagination buttons

        if (test): #In test mode only scrape the first four pages
            pages = range(1,5)

        for page in tqdm(pages, desc=f"Scraping {good[0]} listing pages", unit="page"): #Iterate through the pages
            driver.get(good[1][0]+str(page)+good[1][2]) #Open the page
            htmls.append([good[0],driver.page_source]) #Save the rendered html together with the item category

    driver.close() #Close the webdriver

    return htmls

In [ ]:
metahtml = get_skin_html_metadata() #Run the listing scraper

In [ ]:
def PRC(htmls): #Turn the listing pages into a table with one row per item

    CSS1 ="#__nuxt > div.h-fit.min-h-screen.w-full.bg-theme-900 > div.container.mx-auto.flex-wrap.items-center.px-2.py-4.lg\\:px-6 > div.min-h-screen > div.mx-auto.py-8 > div > div.results-container.lg\\:flex-1 > div:nth-child(3) > div a"
    #CSS selector of the item cards on a listing page

    df = [] #Rows of the output table
    deck = [] #All item cards, each with its item category

    for html in tqdm(htmls, desc = "Processing listing pages",unit = " page"): #Iterate through the listing pages
        cards = BeautifulSoup(html[1],"html.parser").select(CSS1) #Parse the page and select the cards
        deck.extend([[html[0],card] for card in cards]) #Store each card with its item category

    for card in tqdm(deck, desc="Processing item cards", unit = " card"): #Iterate through the cards

        if (card[0]=="knife" or card[0]=="gun"): #Knives and guns
            category = card[0] #Item category
            url = card[1].get("href") #Link to the item page
            name = card[1].find(class_ = "skin-name").get_text() #Skin name
            try: item_type = card[1].find(class_ = "skin-type").get_text() #Weapon type (may include StatTrak or Souvenir)
            except AttributeError: item_type = "" #Missing tag: leave it blank
            try: cond = card[1].find(class_ = "wear-tag").get_text() #Wear condition
            except AttributeError: cond = "" #Missing tag: leave it blank
            try: stock = card[1].find(class_ = "skin-listed-count").get_text() #Number of listings
            except AttributeError: stock = "0" #Missing tag: no listings
            try: price = card[1].find(class_ = "text-md text-theme-200").get_text() #Average listed price
            except AttributeError: price = "0" #Missing tag: no price
            df.append([category,url,name,item_type,cond,stock,price])

        if (card[0]=="case" or card[0]=="key"): #Cases and keys
            category = card[0] #Item category
            url = card[1].get("href") #Link to the item page
            name = card[1].find(class_ = "skin-name").get_text() #Item name
            item_type = "" #Cases and keys have no weapon type
            cond = "" #Cases and keys have no wear condition
            try: stock = card[1].find(class_ = "skin-listed-count").get_text() #Number of listings
            except AttributeError: stock = "0" #Missing tag: no listings
            try: price = card[1].find(class_ = "text-md text-theme-200").get_text() #Average listed price
            except AttributeError: price = "0" #Missing tag: no price
            df.append([category,url,name,item_type,cond,stock,price])

    df = pd.DataFrame(df,columns=["case","url","name","type","cond","stock","price"]) #Build the table ("case" holds the item category)
    df["stock"] = [int(X.replace(" listed","")) for X in df["stock"]] #Number of listings as integer
    df["price"] = [X.replace(",","") for X in df["price"]] #Remove thousand separators
    df["price"] = [float(X.replace("$","")) for X in df["price"]] #Price in USD as number
    df["stattrak"] = ["StatTrak™" in X for X in df["type"]] #StatTrak flag
    df["Souvenir"] = ["Souvenir" in X for X in df["type"]] #Souvenir flag
    df["type"] = [X.replace("StatTrak™ ","") for X in df["type"]] #Remove StatTrak from the type, it now has its own column
    df["type"] = [X.replace("★ ","") for X in df["type"]] #Remove the star symbol
    df["type"] = [X.replace("Souvenir ","") for X in df["type"]] #Remove Souvenir from the type, it now has its own column
    df["scraped"] = False #Whether the item page has been scraped
    df["html"] = None #Html of the item page, filled in section 2
    df["API"] = None #Price-history API link of the item, filled in section 2
    return(df)

In [ ]:
clean = PRC(metahtml) #Build the item table
clean.to_csv(prefix+"metadata.csv",header=True,index= False) #Save it
clean

## 2. Item pages

Each item page is opened in Chrome with performance logging switched on. The page loads its price chart from an internal API endpoint; `proc_log` finds that request in the browser's network log, so the link can be called directly in section 4.

The scraper can resume: items that already have an html are skipped, so after a stop (driver error or bot check) it continues where it left off.

In [ ]:
def proc_log(logs): #Find the price-history API request in the browser's network log
    logs_t = [json.loads(log["message"])["message"]["params"].get("response",{}).get("url","") for log in logs] #Response URLs of all logged network events
    lll = [log for log in logs_t if "https://pricempire.com/api-data/v1/item/chart?id" in log] #Keep the chart API requests
    return(lll[0]) #Return the first one

In [ ]:
def scrape_cards(DF): #Scrape the item pages that have not been scraped yet and return the updated table

    DF = DF.copy()
    DF["scraped"] = DF["html"].notna() #An item counts as scraped if its html is filled in

    got = DF.loc[DF["scraped"]][["url","html","API"]] #Results of earlier runs

    res = [] #Results of this run

    SCS = DF.loc[DF["scraped"]==False]["url"] #Items still to scrape

    if (test): #In test mode only scrape ten item pages
        SCS = SCS[0:10]

    if (len(SCS)>0):
        options = webdriver.ChromeOptions()
        options.set_capability('goog:loggingPrefs', {'performance': 'ALL'}) #Log network events, needed to find the API link
        driver = webdriver.Chrome(options=options)

        for SC in tqdm(SCS,desc="Scraping item pages",unit=" pages"): #Iterate through the remaining items

            try:
                driver.get("https://pricempire.com"+SC) #Open the item page
            except: #If the driver fails, stop and keep what has been collected
                print("Webdriver error, stopping this run")
                break

            try:
                driver.find_element(By.CSS_SELECTOR,".image-shadow") #Check that the item page has loaded (not a bot-check page)
            except: #If not, stop and keep what has been collected
                print("Item page did not load (bot check), stopping this run")
                break

            shtml = driver.page_source #Rendered html of the item page

            try:
                api_url = proc_log(driver.get_log("performance")) #Price-history API link
            except:
                api_url = None #Some items have no price chart

            res.append([SC,shtml,api_url])
            time.sleep(2.5) #Rate limit: wait 2.5 seconds between requests to keep the load on the site low

        try:
            driver.close() #Close the webdriver
        except:
            print("Webdriver was already closed") #The data collected so far is still saved

        res = pd.DataFrame(res,columns=["url","html","API"]) #Results of this run
        res = pd.concat([res,got]) #Add the results of earlier runs

        DF = DF.drop(["html","API"],axis=1) #Drop the old columns so the updated ones can be joined
        DF = DF.join(res.set_index("url"),how = "left",on="url") #Join all results back to the item table

    else:
        print("There is no unscraped item")

    return(DF)

In [ ]:
collect = pd.read_csv(prefix+"metadata.csv") #Item table from section 1

while True: #Repeat until every item page is scraped (in test mode: one run)
    collect = scrape_cards(collect)
    collect.to_parquet(prefix+"scrape.parquet", engine="fastparquet", compression="snappy") #Save progress after every run
    if test or collect["html"].notna().all():
        break
    print("Scraping stopped early, retrying after a 10 s break")
    time.sleep(10)

## 3. Variables from the item pages

The saved html can be large, so the parquet file is read with duckdb in batches of 1,000 items.

In [ ]:
def mass_process(): #Extract the item-page variables from the saved html, in batches

    rec = [] #Processed batches
    files = [file for file in os.listdir() if file.endswith(".parquet") and "scrape" in file and (file.startswith("test_") == test)] #Parquet files of this mode
    con = duckdb.connect()
    con.execute("PRAGMA disable_progress_bar")

    for file in tqdm(files,desc="Processing file",unit="file",position=0): #Iterate through the files

        rows = con.execute("SELECT COUNT(*) FROM '"+file+"' WHERE html IS NOT NULL").fetchone()[0] #Number of scraped items
        R = range(math.ceil(rows/1000)) #Batches of 1,000 items

        for b in tqdm(R,desc="Processing batch",unit="batch",position = 1): #Iterate through the batches
            DF = con.execute("SELECT url, html, API FROM '"+file+"' WHERE html IS NOT NULL LIMIT 1000 OFFSET "+str(1000*b)).df() #Load the current batch

            extra = [] #Extracted variables of the batch

            for J in tqdm(range(len(DF)),desc="Processing item pages",unit="page",position = 2): #Iterate through the items of the batch
                url = DF["url"][J]

                soup = BeautifulSoup(DF["html"][J],"html.parser") #Parse the item page

                try: #Share of sales on western marketplaces (%)
                    west = float(soup.select_one("#market-distribution .p-4+ .p-4 .text-theme-300").get_text().replace("% ",""))
                except:
                    west = None

                try: #Stock listed on each marketplace
                    stock = [x.get_text() for x in soup.select(".text-red-500+ .text-theme-300")]
                    stock = [int(x.replace(" stock","")) for x in stock]
                except:
                    stock = None

                try: #Herfindahl-Hirschman index of the stock across marketplaces (market concentration)
                    HHI = sum([(x/sum(stock))**2 for x in stock])
                except:
                    HHI = None

                try: #Liquidity score of the item (%)
                    liq = float(soup.find(class_="inline-flex items-center text-green-500").get_text().replace("% ",""))
                except:
                    liq = None

                try: #Links of the cases the item can be obtained from
                    case = [x.get("href") for x in soup.select("#containers .hover\\:shadow-lg")]
                except:
                    case = None

                extra.append([url,west,HHI,liq,case])

            extra = pd.DataFrame(extra,columns=["url","west","HHI","liq","caseid"])

            DF = DF.drop(columns=["html"]) #The html is no longer needed
            DF = DF.join(extra.set_index("url"),how = "left",on="url") #Attach the extracted variables
            rec.append(DF)
            gc.collect() #Free memory before the next batch

    return(rec)

In [ ]:
pages = pd.concat(mass_process()) #Process all batches and stack them
pages.to_csv(prefix+"pages.csv",index=False,header=True) #Save the item-page variables
pages

## 4. Price history from the API

The analysis compares prices on 22 and 24 October 2025, before and after the supply shock.

In [ ]:
APIurls = pages["API"].tolist() #API links found in section 2
APIurls = [APIurl for APIurl in APIurls if "https://pricempire.com/api-data/v1/item/chart?id=" in str(APIurl)] #Keep the valid ones

APIscrape = [] #Raw API responses

driver = webdriver.Chrome()
for APIurl in tqdm(APIurls,position = 0): #Iterate through the API links
    try:
        driver.get(APIurl) #Call the API
        APIscrape.append([APIurl,driver.find_element(By.CSS_SELECTOR,"body > pre").text]) #The JSON response is shown inside a <pre> tag
    except:
        print(f"No response, skipping: {APIurl}")
        APIscrape.append([APIurl,"no success"])
    time.sleep(3) #Rate limit: wait 3 seconds between requests
driver.close()

APIdf = pd.DataFrame(APIscrape,columns=["API","data"])
APIdf.to_csv(prefix+"api.csv",header=True,index=False) #Save the raw responses

In [ ]:
extra = [] #Prices and stock at the two dates

timestart = 1761091200 #22 October 2025, 00:00 UTC (UNIX time)
timeend = 1761264000 #24 October 2025, 00:00 UTC (UNIX time)

for I in tqdm(range(len(APIdf))): #Iterate through the API responses

    try: #The response is a list of [time, price, stock] entries
        LIST = pd.DataFrame(ast.literal_eval(APIdf["data"][I]),columns=["time","price","stock"])
    except:
        LIST = None

    url = APIdf["API"][I]

    try: #Price on 22 October
        stpr = int(LIST.loc[LIST["time"]==timestart,"price"].iloc[0])
    except:
        stpr = None

    try: #Stock on 22 October
        stst = int(LIST.loc[LIST["time"]==timestart,"stock"].iloc[0])
    except:
        stst = None

    try: #Price on 24 October
        enpr = int(LIST.loc[LIST["time"]==timeend,"price"].iloc[0])
    except:
        enpr = None

    extra.append([url,stpr,stst,enpr])

extra = pd.DataFrame(extra,columns=["API","stpr","stst","enpr"])
extra

## 5. Final dataset

In [ ]:
meta = pd.read_csv(prefix+"metadata.csv").drop(columns=["html","API","scraped"]) #Listing data from section 1
final = pages.join(extra.set_index("API"),how="left",on="API") #Attach the API prices to the item-page variables
final = meta.join(final.set_index("url"),how="left",on="url") #Attach both to the listing data
final = final.loc[final["stpr"].notna()] #Keep items with a price on 22 October
final.to_csv(prefix+"final.csv",header=True,index=False) #Input of 02_analysis.ipynb
final